# Notebook 2: Summary statistics, ratios and transforming columns
*Kaggle Pandas lesson: "Summary Functions and Maps"*

**Module question: what makes a stock risky?** Today we put the first numbers on it. We summarize
columns, build the ratios that finance runs on (leverage, margins, P/E, price-to-book) and give every
firm a risk label. Along the way we meet a classic trap: the P/E ratio of a company that loses money.

## Learning goals
* summarize a column with `describe`, `mean`, `median`, `unique`, `value_counts`;
* transform a column three ways: `map` with a one-line function, `apply` row by row with `if`/`else`
  logic, and plain arithmetic between columns (the fastest and most common);
* find the row where a column peaks with `idxmax`.

## Setup
Same data as Notebook 1 (one row per US-listed firm above $1B of market value, $ millions; see the
column table there). Each notebook stands alone, so run this cell first.

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/assacohen1/fin6040-pandas-data/main/"
firms = pd.read_csv(DATA_URL + "companies_2025.csv")

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 12)

## In class

### 1. Summary functions
`describe` is the first thing to run on any column. For a numeric column it gives count, mean,
standard deviation, min, quartiles and max.

In [ ]:
firms.vol_2025.describe()

On a text column it gives count, number of distinct values, the most common one and its frequency:

In [ ]:
firms.sector.describe()

It works on several columns at once. Note that `count` is the number of **non-missing** values; when it is smaller than the number of rows, some values are missing.

In [ ]:
firms[["vol_2025", "beta_2025", "ret_2025", "r_and_d"]].describe()

Individual statistics have their own methods:

In [ ]:
firms.vol_2025.mean()

In [ ]:
firms.vol_2025.median()

A mean above the median means a long right tail: a minority of very volatile stocks pulls the average up. Keep that in mind for the exercises.

For categories, `unique` lists the distinct values and `value_counts` counts how often each appears (most frequent first):

In [ ]:
firms.sector.unique()

In [ ]:
firms.sector.value_counts()

In [ ]:
firms.exchange.value_counts()

**Warning.** `value_counts` is for categories. On a column like `vol_2025` every value is different, so
you get a useless list of ones. Use `describe` for numbers.

### 2. `map`: transform every value with a function
`map` runs a function on each value of a Series and returns a new Series. The function is often a
`lambda`: a one-line function without a name. `lambda v: v - vol_mean` means the same as

```python
def center(v):
    return v - vol_mean
```

In [ ]:
vol_mean = firms.vol_2025.mean()
firms.vol_2025.map(lambda v: v - vol_mean)

In [ ]:
def center(v):
    return v - vol_mean

firms.vol_2025.map(center)      # identical result with a named function

`map` returns a **new** Series; `firms` is unchanged unless you assign the result to a column.

### 3. `apply`: transform every row with a function
`apply` with `axis="columns"` hands the function one **row** at a time (as a Series), so the function
can look at several columns and use `if`/`else`. Here is the finance example: the price-earnings ratio
is market value divided by net income, but it only makes sense when net income is positive. A firm that
loses money has no P/E; we return `float("nan")`, the missing-value marker.

In [ ]:
def pe_ratio(row):
    if row.net_income <= 0:
        return float("nan")            # P/E is undefined for a loss-maker
    return row.market_cap / row.net_income

firms["pe"] = firms.apply(pe_ratio, axis="columns")
firms.pe.describe()

`count` is below the number of firms: the loss-makers got NaN, and `describe` (like `mean`, `median`, ...) skips missing values silently.

### 4. Arithmetic between columns (the fast way)
For simple formulas you do not need `map` or `apply`: pandas applies arithmetic to whole columns at
once (this is called *broadcasting*). It is shorter and much faster. The centred volatility again:

In [ ]:
firms.vol_2025 - vol_mean

Now the ratios we will use for the rest of the module:
* **financial leverage** = debt / (debt + market value of equity): the share of the firm financed by debt;
* **debt to assets** = debt / total assets;
* **gross margin** = (sales - cost of goods sold) / sales: high margins usually mean low variable costs;
* **price-to-book** = market value / book value of equity.

In [ ]:
firms["leverage"] = firms.total_debt / (firms.total_debt + firms.market_cap)
firms["debt_to_assets"] = firms.total_debt / firms.total_assets
firms["gross_margin"] = (firms.sales - firms.cogs) / firms.sales     # NaN for banks: no COGS. Notebook 4.
firms["pb"] = firms.market_cap / firms.book_equity
firms.loc[:5, ["ticker", "leverage", "debt_to_assets", "gross_margin", "pe", "pb"]]

Arithmetic cannot express the `if` in `pe_ratio`. Dividing directly gives *negative* P/Es for loss-makers, which is why we needed `apply` above:

In [ ]:
(firms.market_cap / firms.net_income).describe()

Broadcasting works on text too: `+` joins strings column by column.

In [ ]:
firms["label"] = firms.ticker + " (" + firms.sector + ")"
firms.label.head()

### 5. `idxmax`: where is the maximum?
`max` gives the largest value; `idxmax` gives the **index label** of the row where it occurs, which you
can then feed to `loc`. (With the default index the label equals the position, but after `set_index`
it would be a ticker.)

In [ ]:
firms.vol_2025.idxmax()

In [ ]:
firms.loc[firms.vol_2025.idxmax(), ["ticker", "company", "sector", "market_cap", "vol_2025"]]

## Exercises

### Exercise 1: Median volatility

What is the median of `vol_2025`? Assign it to `median_vol`. Is it above or below the mean, and what does that tell you about the shape of the distribution?

<details><summary>Hint</summary>

`.median()` on the column.

</details>

In [ ]:
median_vol = ____

print("median", round(median_vol, 3), " mean", round(firms.vol_2025.mean(), 3))

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert round(median_vol, 4) == 0.3894
print("Looks right!")

### Exercise 2: Which sectors?

Assign the distinct sector names (no duplicates) to `sectors`.

<details><summary>Hint</summary>

`unique()`.

</details>

In [ ]:
sectors = ____

sectors

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert len(sectors) == 11
assert "Utilities" in sectors
print("Looks right!")

### Exercise 3: Firms per sector

Create a Series `firms_per_sector` that maps each sector to its number of firms, largest sector first.

<details><summary>Hint</summary>

`value_counts()` already sorts from most to least frequent.

</details>

In [ ]:
firms_per_sector = ____

firms_per_sector

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert firms_per_sector.sum() == len(firms)
assert len(firms_per_sector) == 11
assert firms_per_sector.index[0] == 'Industrials'
print("Looks right!")

### Exercise 4: Excess volatility

Create `excess_vol`: each firm's `vol_2025` minus the average `vol_2025` across all firms (a centred column). Use arithmetic, not `map`.

<details><summary>Hint</summary>

Column minus a number.

</details>

In [ ]:
excess_vol = ____

excess_vol.describe()

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert excess_vol.shape == (len(firms),)
assert abs(excess_vol.mean()) < 1e-9
print("Looks right!")

### Exercise 5: The cheapest stock

The earnings yield is net income divided by market cap (the inverse of P/E). Which stock has the highest earnings yield? Assign its ticker to `best_value`. Then look at its `pe` and its `ret_2025`: would you buy it on that number alone?

<details><summary>Hint</summary>

Build the ratio Series, take its `idxmax()`, and use that label in `firms.loc[label, "ticker"]`.

</details>

In [ ]:
earnings_yield = ____
best_value = ____

print(best_value)
firms.loc[earnings_yield.idxmax(), ["ticker", "company", "sector", "net_income", "market_cap", "pe", "ret_2025"]]

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert best_value == 'UNIT'
print("Looks right!")

### Exercise 6: Counting with True and False

How many firms have `beta_2025` of at least 1.5, and how many have `vol_2025` of at least 0.5?
Create a Series `risk_counts` with index `["high_beta", "high_vol"]` holding the two counts.
Remember from the Python class that `True + 3` is 4: a True/False Series can be summed.

<details><summary>Hint</summary>

`(firms.beta_2025 >= 1.5).sum()` counts the True values.

</details>

In [ ]:
n_beta = ____
n_vol = ____
risk_counts = pd.Series([n_beta, n_vol], index=["high_beta", "high_vol"])

risk_counts

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert list(risk_counts.index) == ["high_beta", "high_vol"]
assert risk_counts["high_beta"] == 337
assert risk_counts["high_vol"] == 616
print("Looks right!")

### Exercise 7: A risk label for every firm

Write a function `risk_label(row)` that returns `"speculative"` if the firm lost money
(`net_income < 0`), whatever its volatility; otherwise `"high"` if `vol_2025` is at least 0.40,
`"medium"` if it is at least 0.25, and `"low"` otherwise. Apply it row by row to create the column
`firms["risk_label"]`, then show how many firms fall in each label.

<details><summary>Hint</summary>

Same shape as `pe_ratio` above, with `if` / `elif` / `else`. Then `firms.apply(risk_label, axis="columns")`.

</details>

In [ ]:
def risk_label(row):
    ____

firms["risk_label"] = ____
firms.risk_label.value_counts()

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert firms.risk_label.isin(["speculative", "high", "medium", "low"]).all()
assert (firms.risk_label == "speculative").sum() == (firms.net_income < 0).sum()
assert (firms.risk_label == "low").sum() == 249
print("Looks right!")

### Putting it together (worked example)
Does losing money go together with a wild stock price? Compare the average volatility of the two groups.

In [ ]:
loss_vol = firms.loc[firms.net_income < 0, "vol_2025"].mean()
profit_vol = firms.loc[firms.net_income >= 0, "vol_2025"].mean()
share_loss = (firms.net_income < 0).mean()          # mean of True/False = share of True

print(f"Loss-making firms: {share_loss:.0%} of the sample")
print(f"Average volatility:  loss-makers {loss_vol:.2f}   profitable firms {profit_vol:.2f}")

## Finance insight

About 22% of the $1B+ firms lost money in fiscal 2025, and their average volatility
(0.72) is far above that of profitable firms (0.38). Two lessons:

1. **Profitability is itself a risk signal.** The market prices a loss-maker on hopes about the
   future, and hopes swing more than cash flows.
2. **A valuation multiple is only defined when the denominator makes sense.** A negative P/E is not
   "cheap", it is meaningless, and averaging P/Es that include negative ones produces nonsense. That is
   why `pe_ratio` returns NaN for loss-makers. The same care applies to any ratio (EV/EBITDA with
   negative EBITDA, price-to-book with negative equity).

Finally, the mean of `vol_2025` (0.459) sits above the median
(0.389): a minority of very volatile names, mostly loss-makers, biotech and small
tech, creates a long right tail.